# Replication on Kaggle (unattended)

Settings: **Internet on**, **Accelerator: GPU T4 x2**. Add your Hugging Face token as a Kaggle Secret named `HF_TOKEN` (needed for Llama 3.1). Start with **Save Version -> Save & Run All**. Read `replication/README.md` first.

Set the switches in the next cell; each stage is resumable.

In [ ]:
RUN_MULTISALLM = True   # Qwen2.5-Coder-3B-Instruct
RUN_CWEVAL = True       # Llama 3.1 8B Instruct
PLS = 'python'          # 'python', 'python cpp', 'python cpp java'
LANGS = 'English'       # comma separated, or 'all'
CWEVAL_N = 100          # samples per task at T=0.8 (paper: 100)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('No HF_TOKEN secret:', e)
os.environ.update(PLS=PLS, LANGS=LANGS, N=str(CWEVAL_N))
%cd /kaggle/working
!test -d Graduation-Project || git clone -q -b claude/intelligent-noether-sgvujq https://github.com/leegundaniel/Graduation-Project.git
%cd Graduation-Project/replication
!nvidia-smi -L

## Multi-SALLM

In [ ]:
if RUN_MULTISALLM:
    !multisallm/setup.sh
    !multisallm/run.sh refcheck
    !multisallm/run.sh verify-filter
    !multisallm/run.sh published

In [ ]:
if RUN_MULTISALLM:
    !common/serve_vllm.sh start qwen && multisallm/run.sh generate
    !common/serve_vllm.sh stop
    !multisallm/run.sh evaluate

## CWEval

In [ ]:
if RUN_CWEVAL:
    !cweval/setup.sh
    !common/serve_vllm.sh start llama && cweval/run.sh generate
    !common/serve_vllm.sh stop
    !cweval/run.sh evaluate
    !cweval/run.sh compare

## Keep the results
The archive below appears in the notebook's Output tab.

In [ ]:
!cd /kaggle/working/replication_work && tar czf /kaggle/working/replication_results.tgz --exclude='compiled' results runs; ls -lh /kaggle/working/replication_results.tgz
!cat /kaggle/working/replication_work/results/*.md